In [4]:
import MetaTrader5 as mt5
import pandas as pd
import time
import pytz
from datetime import datetime
import numpy as np
import requests
TOKEN = "7227666723:AAEsumQ2gWyr582xK3kGDwMFej0IvX1wD0s"
chat_id = "220684438"

mt5.initialize()


def get_values(symbol):
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_M30, 0, 200)
#     rates_frame = pd.DataFrame(rates)

    rates_frame = pd.DataFrame(rates)
    
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame = rates_frame.drop(['tick_volume', 'spread', 'real_volume',], axis=1)
    rates_frame['rsi1'] = get_rsi(rates_frame['close'], 7)
    rates_frame['rsi2'] = get_rsi(rates_frame['close'], 14)

    # Calculate Supertren
    return rates_frame

def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

def Action_close(ticket_no, symbol, signal, lot):
    try:
        a = [[mt5.symbol_info_tick(symbol).ask, mt5.ORDER_TYPE_BUY], [mt5.symbol_info_tick(symbol).bid, mt5.ORDER_TYPE_SELL]]
        position_id=ticket_no
        price = a[signal][0]
        deviation=1000
        request={
            "action": mt5.TRADE_ACTION_DEAL,    
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][1],
            "position": position_id,
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script close",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result=mt5.order_send(request)
        return result
    except Exception as e:
        print("Action_close_Error")
        print(e)

def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)

def direction(a,j):
    if a.iloc[j].open < a.iloc[j].close:
        return 1
    else:
        return 0
    check = 0

def run(symbol):
    check = 0
    lot = 0.1
    buy_check = 0
    sell_check = 0
    buy_up = 0
    sell_up = 0
    order_time = 0
    old = 0
    old_pp = 0

    buy = 1
    sell = 0
 
    print(symbol)
    hour_passed = True

    while True:
        a = get_values(symbol)
        if a.iloc[-2].close != old:
            if a.iloc[-3].rsi1 > 70 and a.iloc[-3].rsi1 > a.iloc[-2].rsi1:
                message = f"Sell USDCAD --> {a.iloc[-1].name}"
                print(a.iloc[-2])
                print(f"close -- {a.iloc[-2].close} ## ema1-- ## {a.iloc[-2].name}")
                url = f"https://api.telegram.org/bot{TOKEN}/sendMessage?chat_id={chat_id}&text={message}"
                r = requests.get(url)
                print(r.json())
                old = a.iloc[-2].close
                sell_check = 0
                
                
            elif a.iloc[-3].rsi1 < 30 and a.iloc[-3].rsi1 < a.iloc[-2].rsi1:
                    message = f"BUY USDCAD --> {a.iloc[-1].name}"
                    url = f"https://api.telegram.org/bot{TOKEN}/sendMessage?chat_id={chat_id}&text={message}"
                    r = requests.get(url)
                    print(r.json())
                    old = a.iloc[-2].close

for symbol in ['USDCAD']:
    run(symbol)
    

USDCAD
{'ok': True, 'result': {'message_id': 238, 'from': {'id': 7227666723, 'is_bot': True, 'first_name': 'signal', 'username': 'usdema_bot'}, 'chat': {'id': 220684438, 'first_name': 'Animesh', 'last_name': 'Verma', 'username': 'xicor', 'type': 'private'}, 'date': 1726657201, 'text': 'BUY USDCAD --> 2024-09-18 14:00:00'}}
open      1.360640
high      1.360680
low       1.359470
close     1.359500
rsi1     54.717671
rsi2     54.382431
Name: 2024-09-18 19:00:00, dtype: float64
close -- 1.3595 ## ema1-- ## 2024-09-18 19:00:00
{'ok': True, 'result': {'message_id': 239, 'from': {'id': 7227666723, 'is_bot': True, 'first_name': 'signal', 'username': 'usdema_bot'}, 'chat': {'id': 220684438, 'first_name': 'Animesh', 'last_name': 'Verma', 'username': 'xicor', 'type': 'private'}, 'date': 1726677000, 'text': 'Sell USDCAD --> 2024-09-18 19:30:00'}}
{'ok': True, 'result': {'message_id': 240, 'from': {'id': 7227666723, 'is_bot': True, 'first_name': 'signal', 'username': 'usdema_bot'}, 'chat': {'id':

KeyboardInterrupt: 